<a href="https://colab.research.google.com/github/junkyuhufs/Class2026spring/blob/main/NLPforET_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pandas + Text Analysis for English Teachers 🌸📚


오늘은 아래 흐름으로 진행합니다.

### Part 1. pandas와 시각화 감 잡기
- 숫자형 표 데이터(Iris) 보기

### Part 2. 텍스트를 변수로 바꿔 pandas로 보기
- 웹에서 텍스트 불러오기
- 문단별 특징(feature) 추출하기
- 표와 그래프로 보기

### Part 3. 간단한 NLP 확장
- 자주 나오는 단어
- word cloud
- 문장 길이
- concordance

---

## 오늘의 핵심 목표 🎯
1. pandas는 표 데이터를 다루는 도구라는 것  
2. 텍스트도 변수로 바꿔 표처럼 다룰 수 있다는 것  
3. 리딩 텍스트를 간단한 NLP 방식으로 분석할 수 있다는 것

In [ ]:
# 필요한 라이브러리 설치
!pip -q install pandas matplotlib scikit-learn nltk textstat wordcloud requests

In [ ]:
# 라이브러리 불러오기
import pandas as pd
import matplotlib.pyplot as plt
import requests
import nltk

from sklearn.datasets import load_iris
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.text import Text
from collections import Counter
from wordcloud import WordCloud
from textstat import flesch_reading_ease

In [ ]:
# NLTK 자료 다운로드
nltk.download("punkt")
nltk.download("stopwords")

# Part 1. pandas와 시각화 감 잡기 🌸

먼저 숫자형 표 데이터를 보겠습니다.  
오늘은 유명한 Iris 데이터를 사용합니다.

In [ ]:
# Iris 데이터 불러오기
iris = load_iris()

# DataFrame 만들기
df_iris = pd.DataFrame(iris.data, columns=iris.feature_names)
df_iris["species"] = [iris.target_names[i] for i in iris.target]

# 앞부분 보기
df_iris.head()

### 결과가 의미하는 것 💡
이 표의 각 행은 꽃 한 개, 각 열은 꽃의 측정값입니다.  
즉, pandas는 이런 표를 다루는 도구입니다.

In [ ]:
# 데이터 구조 확인
print(df_iris.shape)      # 행, 열 개수
print(df_iris.columns)    # 열 이름

In [ ]:
# species별 평균 보기
df_iris.groupby("species").mean(numeric_only=True)

### 결과가 의미하는 것 💡
이 표는 종(species)별 평균 비교표입니다.  
즉, 그룹별 비교의 가장 기본적인 예입니다.

In [ ]:
# species별 평균 petal length 막대그래프
species_means = df_iris.groupby("species")["petal length (cm)"].mean()

plt.figure(figsize=(8, 4))
plt.bar(species_means.index, species_means.values)
plt.xlabel("Species")
plt.ylabel("Average Petal Length")
plt.title("Average Petal Length by Species")
plt.show()

In [ ]:
# sepal length 분포 히스토그램
plt.figure(figsize=(8, 4))
plt.hist(df_iris["sepal length (cm)"], bins=15)
plt.xlabel("Sepal Length")
plt.ylabel("Frequency")
plt.title("Distribution of Sepal Length")
plt.show()

### Part 1 정리 ✅
여기서 핵심은:
- pandas는 표 데이터를 다룬다
- 평균을 비교할 수 있다
- 그래프로 데이터를 더 쉽게 볼 수 있다

# Part 2. 텍스트를 변수로 바꿔 pandas로 보기 📚

이제 텍스트를 그냥 읽는 대신,  
문단별 특징을 뽑아서 표로 만들어 보겠습니다.

In [ ]:
# 웹에서 Alice in Wonderland 텍스트 불러오기
url = "https://www.gutenberg.org/files/11/11-0.txt"
raw_text = requests.get(url).text

# 앞부분 확인
print(raw_text[:800])

In [ ]:
# Gutenberg 안내문 제거
start_marker = "*** START OF THE PROJECT GUTENBERG EBOOK"
end_marker = "*** END OF THE PROJECT GUTENBERG EBOOK"

start_idx = raw_text.find(start_marker)
end_idx = raw_text.find(end_marker)

clean_text = raw_text[start_idx:end_idx] if start_idx != -1 and end_idx != -1 else raw_text

print(clean_text[:800])

In [ ]:
# 문단 나누기
paragraphs = [p.strip() for p in clean_text.split("\n\n") if len(p.strip()) > 100]

print("Number of paragraphs:", len(paragraphs))
print(paragraphs[0][:300])

### 결과가 의미하는 것 💡
긴 텍스트 하나를 여러 문단으로 나눈 것입니다.  
이제 문단 하나하나를 분석 단위로 볼 수 있습니다.

In [ ]:
# 문단별 특징 추출 함수 만들기
def extract_text_features(paragraph):
    sentences = sent_tokenize(paragraph)
    words = [w.lower() for w in word_tokenize(paragraph) if w.isalpha()]

    sentence_count = len(sentences)
    word_count = len(words)
    unique_words = len(set(words))
    avg_sentence_length = word_count / sentence_count if sentence_count > 0 else 0
    lexical_diversity = unique_words / word_count if word_count > 0 else 0
    readability = flesch_reading_ease(paragraph)

    return {
        "sentence_count": sentence_count,
        "word_count": word_count,
        "avg_sentence_length": avg_sentence_length,
        "lexical_diversity": lexical_diversity,
        "readability": readability
    }

In [ ]:
# 앞 10개 문단만 분석
feature_list = []

for i, p in enumerate(paragraphs[:10]):
    features = extract_text_features(p)
    features["paragraph_id"] = i + 1
    feature_list.append(features)

df_text = pd.DataFrame(feature_list)
df_text

### 결과가 의미하는 것 💡
이제 각 행은 문단 하나이고,  
각 열은 그 문단의 특징입니다.

즉, 텍스트를 **표 데이터로 바꾼 것**입니다.

In [ ]:
# 문단별 단어 수 그래프
plt.figure(figsize=(10, 4))
plt.bar(df_text["paragraph_id"], df_text["word_count"])
plt.xlabel("Paragraph ID")
plt.ylabel("Word Count")
plt.title("Word Count by Paragraph")
plt.show()

In [ ]:
# 문단별 lexical diversity 그래프
plt.figure(figsize=(10, 4))
plt.plot(df_text["paragraph_id"], df_text["lexical_diversity"], marker="o")
plt.xlabel("Paragraph ID")
plt.ylabel("Lexical Diversity")
plt.title("Lexical Diversity by Paragraph")
plt.show()

In [ ]:
# 문단별 readability 그래프
plt.figure(figsize=(10, 4))
plt.bar(df_text["paragraph_id"], df_text["readability"])
plt.xlabel("Paragraph ID")
plt.ylabel("Readability")
plt.title("Readability by Paragraph")
plt.show()

### Part 2 정리 ✅
여기서 중요한 것은:
- 텍스트도 데이터다 (비정형 데이터)
- 문단별 특징을 숫자로 만들 수 있다
- 그 숫자들을 pandas로 정리하고 시각화할 수 있다

# Part 3. 간단한 NLP 확장 🔎

이제 같은 텍스트를 조금 더 NLP답게 분석해 보겠습니다.

오늘은 아래 4가지만 봅니다.

1. 자주 나오는 단어  
2. word cloud  
3. 문장 길이 분포  
4. concordance

In [ ]:
# 텍스트 전체를 단어로 나누기
tokens = word_tokenize(clean_text)
words = [w.lower() for w in tokens if w.isalpha()]

# stopwords 제거
stop_words = set(stopwords.words("english"))
content_words = [w for w in words if w not in stop_words]

In [ ]:
# 자주 나오는 단어 15개 보기
freq = Counter(content_words)
top15 = freq.most_common(15)

pd.DataFrame(top15, columns=["word", "frequency"])

### 결과가 의미하는 것 💡
이 표는 텍스트에서 핵심적으로 반복되는 단어를 보여줍니다.  
즉, reading passage의 핵심 어휘를 빠르게 파악할 수 있습니다.

In [ ]:
# word cloud 만들기
wordcloud = WordCloud(width=1000, height=500, background_color="white").generate(" ".join(content_words))

plt.figure(figsize=(14, 6))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("Word Cloud")
plt.show()

In [ ]:
# 문장 길이 계산
sentences = sent_tokenize(clean_text)

sentence_lengths = []
for sent in sentences:
    sent_words = [w for w in word_tokenize(sent) if w.isalpha()]
    sentence_lengths.append(len(sent_words))

print("Average sentence length:", round(sum(sentence_lengths) / len(sentence_lengths), 2))

In [ ]:
# 문장 길이 분포 그래프
plt.figure(figsize=(10, 5))
plt.hist(sentence_lengths, bins=20)
plt.xlabel("Sentence Length (in words)")
plt.ylabel("Frequency")
plt.title("Sentence Length Distribution")
plt.show()

### 결과가 의미하는 것 💡
이 그래프는 문장 길이가 대체로 짧은지 긴지 보여줍니다.  
즉, 텍스트의 읽기 부담을 생각하는 데 도움이 됩니다.

In [ ]:
# concordance 보기
alice_text_obj = Text(words)

alice_text_obj.concordance("alice", lines=10)

### 결과가 의미하는 것 💡
concordance는 특정 단어가 실제 문맥에서 어떻게 쓰이는지 보여줍니다.  
즉, 단어를 뜻만이 아니라 **맥락 속에서** 보는 방법입니다.

# 전체 수업 정리 ✅

오늘 수업의 흐름은 이렇습니다.

### Part 1
숫자형 데이터로 pandas와 시각화 감 잡기 🌸

### Part 2
텍스트를 변수로 바꿔 pandas DataFrame 만들기 📚

### Part 3
리딩 텍스트를 간단한 NLP 방식으로 분석하기 🔎

---

## 오늘 핵심
- pandas는 표 데이터를 다루는 도구이다
- 텍스트도 변수로 바꿔 표처럼 다룰 수 있다
- 리딩 텍스트는 다양한 방식으로 분석할 수 있다
- 영어교사도 이런 분석을 수업 자료 해석에 활용할 수 있다